# Notebook 03 - Statistics and EDA (M4)

Covers: M4.1 Descriptive statistics, M4.2 Probability/distributions, M4.3 Inferential statistics, M4.4 Hypothesis tests, M4.5 EDA

Syllabus: Unit II, III, IV

In [ ]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from src.utils.helpers import setup_logging
setup_logging()
sns.set_theme(style="whitegrid")
FIGURES = "../reports/figures"
os.makedirs(FIGURES, exist_ok=True)
print("Environment ready.")
from scipy import stats
from scipy.stats import skew, kurtosis as kurt
from src.utils.constants import FACTORS_TIDY, FACTORS_WIDE, PLACES_CSV
tidy = pd.read_parquet(FACTORS_TIDY)
wide = pd.read_csv(FACTORS_WIDE)
places = pd.read_csv(PLACES_CSV)
wide = wide.merge(places[['place_id','region','place_type','state_name']], on='place_id', how='left')
pm25 = tidy[(tidy['factor']=='air_quality') & (tidy['value_raw'].notna())].copy()
pm25 = pm25.merge(places[['place_id','region','place_type']], on='place_id', how='left')
pm25['year'] = pm25['year'].astype('Int64')
pm25_vals = pm25['value_raw'].dropna()
print(f'Tidy: {tidy.shape}, PM2.5 rows: {len(pm25_vals)}')

## 4.1 Descriptive Statistics

Mean, median, mode, variance, SD, range, IQR, skewness, kurtosis per factor.

In [ ]:
desc_rows = []
for factor, grp in tidy.groupby('factor'):
    vals = grp['value_raw'].dropna()
    if len(vals) == 0: continue
    desc_rows.append({'factor':factor,'n':len(vals),'mean':vals.mean(),'median':vals.median(),'mode':vals.mode().iloc[0] if len(vals.mode())>0 else np.nan,'variance':vals.var(),'std':vals.std(),'range':vals.max()-vals.min(),'IQR':vals.quantile(0.75)-vals.quantile(0.25),'skewness':skew(vals),'kurtosis':kurt(vals),'min':vals.min(),'max':vals.max()})
desc_df = pd.DataFrame(desc_rows).set_index('factor')
pd.set_option('display.float_format', '{:.3f}'.format)
display(desc_df)

### Correlation matrix (Spearman)

In [ ]:
factor_cols = [c for c in wide.columns if c in ['pm25_mean','crime_rate_per_lakh','infant_mortality_rate','per_capita_nsdp_constant','unemployment_rate','avg_rent']]
if factor_cols:
    corr = wide[factor_cols].corr(method='spearman')
    fig, ax = plt.subplots(figsize=(8,6))
    sns.heatmap(corr, ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1, annot=True, fmt='.2f', linewidths=0.5)
    ax.set_title('Spearman Correlation Matrix')
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/03_correlation_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No numeric factor columns found in wide table.')

## 4.2 Probability and Distributions

Normality test for PM2.5 using histogram, Q-Q plot, Shapiro-Wilk.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
fig.suptitle('PM2.5 Distribution Analysis', fontsize=13, fontweight='bold')
mu, sigma = pm25_vals.mean(), pm25_vals.std()
axes[0].hist(pm25_vals, bins=40, density=True, color='#3498db', edgecolor='white', alpha=0.8, label='Observed')
x = np.linspace(pm25_vals.min(), pm25_vals.max(), 300)
axes[0].plot(x, stats.norm.pdf(x, mu, sigma), 'r-', lw=2, label=f'Normal(mu={mu:.1f})')
axes[0].set_xlabel('PM2.5 (ug/m3)')
axes[0].set_title('Histogram + Normal fit')
axes[0].legend(fontsize=8)
stats.probplot(pm25_vals, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot')
log_pm = np.log1p(pm25_vals)
axes[2].hist(log_pm, bins=40, density=True, color='#2ecc71', edgecolor='white', alpha=0.8)
ml, sl = log_pm.mean(), log_pm.std()
xl = np.linspace(log_pm.min(), log_pm.max(), 300)
axes[2].plot(xl, stats.norm.pdf(xl, ml, sl), 'r-', lw=2)
axes[2].set_xlabel('log(PM2.5+1)')
axes[2].set_title('Log-transformed')
plt.tight_layout()
plt.savefig(f'{FIGURES}/03_pm25_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
stat_sw, p_sw = stats.shapiro(pm25_vals.sample(min(500, len(pm25_vals)), random_state=42))
print(f'Shapiro-Wilk (raw PM2.5): W={stat_sw:.4f}, p={p_sw:.4e}')
stat_sl, p_sl = stats.shapiro(log_pm.sample(min(500, len(log_pm)), random_state=42))
print(f'Shapiro-Wilk (log PM2.5): W={stat_sl:.4f}, p={p_sl:.4e}')
print(f'PM2.5 normal: {p_sw >= 0.05}. Log-normal: {p_sl >= 0.05}')

## 4.3 Inferential Statistics

Central Limit Theorem demo and 95% confidence interval.

In [ ]:
n_draws, n_samp = 1000, 30
samp_means = np.array([pm25_vals.sample(n_samp, random_state=i).mean() for i in range(n_draws)])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(pm25_vals, bins=50, density=True, color='#e74c3c', alpha=0.7, label='Population')
axes[0].set_title('Population distribution (skewed)')
axes[0].legend()
axes[1].hist(samp_means, bins=40, density=True, color='#3498db', alpha=0.8, label=f'Sample means (n={n_samp})')
xs = np.linspace(samp_means.min(), samp_means.max(), 300)
axes[1].plot(xs, stats.norm.pdf(xs, samp_means.mean(), samp_means.std()), 'r-', lw=2)
axes[1].set_title(f'Distribution of {n_draws} sample means (CLT)')
axes[1].legend(fontsize=8)
plt.suptitle('Central Limit Theorem Demo', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES}/03_clt.png', dpi=150, bbox_inches='tight')
plt.show()
n = len(pm25_vals)
ci_lo, ci_hi = stats.t.interval(0.95, df=n-1, loc=pm25_vals.mean(), scale=pm25_vals.sem())
print(f'95% CI for mean PM2.5: [{ci_lo:.2f}, {ci_hi:.2f}] ug/m3')

## 4.4 Hypothesis Tests

State H0 and H1, check assumptions, interpret p-values.

In [ ]:
from scipy.stats import ttest_ind, f_oneway, ttest_1samp, spearmanr
print('H1: Mean PM2.5 North vs South (Welch t-test)')
print('H0: mu_north = mu_south | H1: mu_north != mu_south')
north = pm25[pm25['region']=='North']['value_raw'].dropna()
south = pm25[pm25['region']=='South']['value_raw'].dropna()
if len(north)>5 and len(south)>5:
    t1, p1 = ttest_ind(north, south, equal_var=False)
    print(f'North mean: {north.mean():.2f}, South mean: {south.mean():.2f}')
    print(f't={t1:.4f}, p={p1:.4e}')
    print(f'Decision: {"REJECT H0" if p1<0.05 else "FAIL TO REJECT H0"}')
    print('Type I error: wrongly rejecting H0 (controlled at alpha=0.05)')
    print('Type II error: missing a real difference (increases with small n)')
else:
    print('Insufficient data.')

In [ ]:
print('H2: PM2.5 differs across regions? (one-way ANOVA)')
print('H0: All region means equal | H1: At least one differs')
groups = [pm25[pm25['region']==r]['value_raw'].dropna().tolist() for r in pm25['region'].dropna().unique() if len(pm25[pm25['region']==r])>2]
if len(groups)>=3:
    F, p_an = f_oneway(*groups)
    print(f'F={F:.4f}, p={p_an:.4e}')
    print(f'Decision: {"REJECT H0" if p_an<0.05 else "FAIL TO REJECT H0"}')
    if p_an<0.05: print('Post-hoc Tukey HSD would identify which regions differ.')

In [ ]:
NAAQS=60.0
print(f'H3: Mean PM2.5 > NAAQS standard ({NAAQS} ug/m3)? (one-sample t-test)')
print(f'H0: mu={NAAQS} | H1: mu>{NAAQS}')
t3, p3 = ttest_1samp(pm25_vals, NAAQS)
p3_os = p3/2 if t3>0 else 1-p3/2
print(f'Sample mean: {pm25_vals.mean():.2f} ug/m3')
print(f't={t3:.4f}, p(one-sided)={p3_os:.4e}')
print(f'Decision: {"REJECT H0" if p3_os<0.05 else "FAIL TO REJECT H0"} at alpha=0.05')

## 4.5 EDA Visualizations

In [ ]:
# Box plots per factor
fig, axes = plt.subplots(1, len(desc_df), figsize=(14,5))
fig.suptitle('Factor Distributions (Box Plots)', fontsize=13, fontweight='bold')
for ax, factor in zip(axes if len(desc_df)>1 else [axes], desc_df.index):
    vals = tidy[tidy['factor']==factor]['value_raw'].dropna()
    ax.boxplot(vals, patch_artist=True, boxprops=dict(facecolor='#3498db', alpha=0.7))
    ax.set_title(factor.replace('_',' ').title(), fontsize=9)
    ax.tick_params(axis='x', labelbottom=False)
plt.tight_layout()
plt.savefig(f'{FIGURES}/03_box_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Time series: PM2.5 trend per city (top 10)
top_cities = pm25.groupby('place_id')['year'].count().nlargest(10).index
fig, ax = plt.subplots(figsize=(12,5))
for pid in top_cities:
    sub = pm25[pm25['place_id']==pid].sort_values('year')
    nm = places[places['place_id']==pid]['place_name'].values
    name = nm[0] if len(nm)>0 else pid
    ax.plot(sub['year'], sub['value_raw'], marker='o', markersize=4, label=name)
ax.axvline(2020, color='gray', ls='--', alpha=0.6, label='COVID 2020')
ax.axhline(60, color='red', ls=':', alpha=0.7, label='NAAQS 60 ug/m3')
ax.set_xlabel('Year')
ax.set_ylabel('Annual Mean PM2.5 (ug/m3)')
ax.set_title('PM2.5 Trend: Top 10 Cities by Data Coverage')
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(f'{FIGURES}/03_pm25_time_series.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Regional box plot
if 'region' in pm25.columns:
    order = pm25.groupby('region')['value_raw'].median().sort_values(ascending=False).index
    fig, ax = plt.subplots(figsize=(10,5))
    sns.boxplot(data=pm25, x='region', y='value_raw', order=order, ax=ax, palette='Set2')
    ax.axhline(60, color='red', ls='--', alpha=0.8, label='NAAQS')
    ax.set_ylabel('PM2.5 (ug/m3)')
    ax.set_title('PM2.5 Distribution by Region')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/03_pm25_by_region.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
import pathlib
pathlib.Path('../reports').mkdir(exist_ok=True)
findings = [f'Mean PM2.5: {pm25_vals.mean():.2f} ug/m3', f'PM2.5 normally distributed (SW p): {p_sw:.4e}', f'North mean PM2.5: {north.mean():.2f}', f'South mean PM2.5: {south.mean():.2f}']
with open('../reports/findings.md', 'w') as f:
    f.write('# Key Statistical Findings\n\n')
    for line in findings: f.write(f'- {line}\n')
print('Saved: reports/findings.md')